In [3]:
import numpy as np
import pandas  as pd
import tensorflow as tf
print(tf.__version__)

2.21.0


In [4]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
if "RowNumber" in data.columns:
    data=data.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [6]:
print(data.columns)

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='str')


In [7]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
data["Geography"]=le.fit_transform(data["Geography"])
data["Gender"]=le.fit_transform(data["Gender"])
X=data.drop(["Exited"],axis= 1)
y=data["Exited"]

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=0)

In [9]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

ann intialization (making an model with the help of tensorflow intialization  )

In [10]:
from tensorflow.keras import metrics
ann=tf.keras.models.Sequential()
ann.add(tf.keras.layers.Dense(units=6,activation="relu"))
ann.add(tf.keras.layers.Dense(units=6,activation="relu"))
ann.add(tf.keras.layers.Dense(units=1,activation="sigmoid"))
ann.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])

In [11]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor ="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
monitor='val_loss',
factor=0.2,
patience=10,
min_lr=0.0001
)

In [12]:

history = ann.fit(
X_train, y_train,
batch_size=32,
epochs=100,
validation_data=(X_test, y_test))

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6555 - loss: 0.6091 - val_accuracy: 0.7810 - val_loss: 0.5085
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 870us/step - accuracy: 0.7947 - loss: 0.4728 - val_accuracy: 0.7965 - val_loss: 0.4607
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 921us/step - accuracy: 0.8064 - loss: 0.4466 - val_accuracy: 0.8080 - val_loss: 0.4403
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 905us/step - accuracy: 0.8126 - loss: 0.4342 - val_accuracy: 0.8155 - val_loss: 0.4288
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 949us/step - accuracy: 0.8181 - loss: 0.4266 - val_accuracy: 0.8220 - val_loss: 0.4205
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 952us/step - accuracy: 0.8194 - loss: 0.4201 - val_accuracy: 0.8275 - val_loss: 0.4124
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 909us/step - accuracy: 0.8273 - loss: 0.4135 - val_accuracy: 0.8365 - val_loss: 0.4037
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 895us/step - accuracy: 0.8311 - loss: 0.4

In [13]:
print(ann.predict(sc.transform([[1, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])) > 0.5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
[[False]]


c:\Users\JEFF JOSEPH\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [14]:
from sklearn.metrics import classification_report
y_pred = ann.predict(X_test)
y_pred = (y_pred > 0.5)
print(classification_report(y_test,y_pred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 746us/step
              precision    recall  f1-score   support

           0       0.88      0.96      0.92      1595
           1       0.76      0.48      0.59       405

    accuracy                           0.86      2000
   macro avg       0.82      0.72      0.76      2000
weighted avg       0.86      0.86      0.85      2000



In [16]:
from sklearn.metrics import accuracy_score,confusion_matrix
cm=confusion_matrix(y_test,y_pred)
print(cm)
print(accuracy_score(y_test,y_pred))

[[1534   61]
 [ 209  196]]
0.865


evaluvating my trained ann set (Checking accuracy and error rate)

In [ ]:
print("\n Evaluvating the model...")
train_loss,train_acc=ann.evaluate(X_train,y_train,verbose=0)
test_loss,test_acc=ann.evaluate(X_test,y_test,verbose=0)

y_train_pred=ann.predict(X_train)
y_train_pred_class=(y_train_pred>0.5)
